__Inputs__:

source: https://www.bls.gov/cps/tables.htm

Download and save as:
`data/bls/cpsaat11_{year}.xlsx` (2015 to 2024)

__Outputs__:

`data/bls/bls_industry_data_{year}.csv`: Job-category level employment data

`data/bls/bls_occupation_data_{year}.csv`: Occupation level employment data

`data/bls/bls_combined_data_{year}.csv`: Occupation level employment with parent categories

`data/occupations_{year}.txt`: Unique list of occupations by year for LLMs


In [ ]:
"""
Parse BLS CPS Table 11 (Employed persons by detailed occupation) into
normalized DataFrames that separate the hierarchy from the leaf-level data.

The Excel file encodes hierarchy via cell indent levels:
  indent 0 = major group  (e.g. "Management, professional, and related occupations")
  indent 1 = mid group    (e.g. "Management, business, and financial operations occupations")
  indent 2 = minor group  (e.g. "Management occupations")
  indent 3 = detail       (e.g. "Chief executives")

Some sections skip levels (e.g. indent 1 -> leaves at indent 2 with no indent 3).
A row is a "leaf" if the next occupation row has the same or lower indent level.

Returns:
  - `leaves`: one row per leaf occupation with data columns + a `parent_id` FK
  - `hierarchy`: one row per parent occupation with `id`, `name`, `level`, `parent_id`

These can be joined: leaves -> hierarchy -> hierarchy -> ... to walk up the tree.
"""

import numpy as np
import pandas as pd
from openpyxl import load_workbook


DATA_COLS = [
    "total_employed", "pct_women", "pct_white",
    "pct_black", "pct_asian", "pct_hispanic",
]


def _to_numeric(val):
    """Convert a cell value to float, treating dashes and blanks as NaN."""
    if val is None:
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).strip()
    if s in ("–", "-", "—", ""):
        return np.nan
    return float(s)


def parse_bls_occupation_file(filepath: str, dta_type: str = "occupation") -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parse a BLS CPS Table 11 xlsx file into leaf data and hierarchy tables.

    Parameters
    ----------
    filepath : str
        Path to the .xlsx file (e.g. cpsaat11_2024.xlsx).

    Returns
    -------
    leaves : pd.DataFrame
        Columns: occupation, total_employed, pct_women, pct_white,
                 pct_black, pct_asian, pct_hispanic, parent_id
    hierarchy : pd.DataFrame
        Columns: id, occupation, level, parent_id, total_employed,
                 pct_women, pct_white, pct_black, pct_asian, pct_hispanic
    """
    wb = load_workbook(filepath)
    ws = wb.active

    # --- Step 1: Extract all occupation rows with indent + data ---
    raw_rows = []
    for row_idx in range(1, ws.max_row + 1):
        cell = ws.cell(row=row_idx, column=1)
        val = cell.value
        if not val or not isinstance(val, str) or not val.strip():
            continue
        name = " ".join(val.split())
        # Skip header/footer rows
        if name.startswith(("HOUSEHOLD", "[Numbers", "Occupation", "Industry", "NOTE:", "See", "n.e.c")):
            continue
        if name == "Total, 16 years and over":
            continue  # overall total, not part of hierarchy

        indent = int(cell.alignment.indent) if cell.alignment else 0
        data = [_to_numeric(ws.cell(row=row_idx, column=c).value) for c in range(2, 8)]
        raw_rows.append({
            "row_idx": row_idx,
            "indent": indent,
            dta_type: name,
            "total_employed": data[0],
            "pct_women": data[1],
            "pct_white": data[2],
            "pct_black": data[3],
            "pct_asian": data[4],
            "pct_hispanic": data[5],
        })

    # --- Step 2: Classify each row as parent or leaf ---
    # A parent is any row whose next row has a strictly higher indent.
    for i, row in enumerate(raw_rows):
        if i < len(raw_rows) - 1 and raw_rows[i + 1]["indent"] > row["indent"]:
            row["is_parent"] = True
        else:
            row["is_parent"] = False

    # --- Step 3: Assign IDs to parents and build hierarchy ---
    # Walk through rows maintaining a stack of ancestors (one per indent level).
    # Stack maps indent_level -> parent dict.
    parent_id_counter = 0
    ancestor_stack = {}  # indent_level -> parent row dict
    hierarchy_rows = []

    for row in raw_rows:
        indent = row["indent"]

        if row["is_parent"]:
            parent_id_counter += 1
            row["id"] = f"L{indent}_{parent_id_counter:04d}"

            # This parent's own parent is the most recent ancestor
            # at a strictly lower indent level.
            my_parent_id = None
            for lvl in sorted(ancestor_stack.keys(), reverse=True):
                if lvl < indent:
                    my_parent_id = ancestor_stack[lvl]["id"]
                    break
            row["parent_id"] = my_parent_id

            # Clear any ancestor entries at this level or deeper
            ancestor_stack = {k: v for k, v in ancestor_stack.items() if k < indent}
            ancestor_stack[indent] = row

            hierarchy_rows.append({
                "id": row["id"],
                dta_type: row[dta_type],
                "level": indent,
                "parent_id": my_parent_id,
                "total_employed": row["total_employed"],
                "pct_women": row["pct_women"],
                "pct_white": row["pct_white"],
                "pct_black": row["pct_black"],
                "pct_asian": row["pct_asian"],
                "pct_hispanic": row["pct_hispanic"],
            })
        else:
            # Leaf: find its parent from the ancestor stack
            leaf_parent_id = None
            for lvl in sorted(ancestor_stack.keys(), reverse=True):
                if lvl < indent:
                    leaf_parent_id = ancestor_stack[lvl]["id"]
                    break
            row["parent_id"] = leaf_parent_id

    # --- Step 4: Build DataFrames ---
    leaf_rows = [r for r in raw_rows if not r["is_parent"]]

    leaves = pd.DataFrame(leaf_rows)[[
        dta_type, "total_employed", "pct_women", "pct_white",
        "pct_black", "pct_asian", "pct_hispanic", "parent_id", "indent"
    ]]

    hierarchy = pd.DataFrame(hierarchy_rows)[[
        "id", dta_type, "level", "parent_id", "total_employed",
        "pct_women", "pct_white", "pct_black", "pct_asian", "pct_hispanic"
    ]]

    return leaves, hierarchy


def get_full_ancestry(leaves: pd.DataFrame, hierarchy: pd.DataFrame, dta_type: str = "occupation") -> pd.DataFrame:
    """
    Convenience: join leaves with all ancestor levels to produce a wide table.

    Returns a DataFrame with columns:
        occupation, [data cols], parent_occupation, grandparent_occupation, ...
    up to the number of hierarchy levels present.
    """
    # Build a lookup from hierarchy id -> row
    h = hierarchy.set_index("id")
    max_depth = int(hierarchy["level"].max()) + 1

    ancestor_cols = {i: [] for i in range(max_depth)}

    for _, leaf in leaves.iterrows():
        pid = leaf["parent_id"]
        visited = {}
        while pid and pid in h.index:
            parent = h.loc[pid]
            visited[int(parent["level"])] = parent[dta_type]
            pid = parent["parent_id"]
        for lvl in range(max_depth):
            ancestor_cols[lvl].append(visited.get(lvl, None))

    result = leaves.copy()
    level_names = {0: "major_group", 1: "mid_group", 2: "minor_group"}
    for lvl in range(max_depth):
        col_name = level_names.get(lvl, f"level_{lvl}")
        result[col_name] = ancestor_cols[lvl]

    return result


In [2]:
for year in range(2015, 2025):
    fp = f"../data/bls/cpsaat11_{year}.xlsx"
    leaves, hierarchy = parse_bls_occupation_file(fp)
    leaves.to_csv(f"../data/bls/bls_occupation_data_{year}.csv", index=False)
    hierarchy.to_csv(f"../data/bls/bls_industry_data_{year}.csv", index=False)
    
    full = get_full_ancestry(leaves, hierarchy)
    full.to_csv(f"../data/bls/bls_combined_data_{year}.csv", index=False)

    all_occupations = sorted(set(leaves["occupation"].tolist()))
    all_industries = sorted(set(hierarchy["occupation"].to_list()))
    
    with open(f"../data/occupations_{year}.txt", "w") as f:
        f.write("\n".join(all_occupations))

    with open(f"../data/industry_{year}.txt", "w") as f:
        f.write("\n".join(all_industries))



In [ ]:
for year in range(2015, 2025):
    fp = f"../data/bls/cpsaat18_{year}.xlsx"
    leaves, hierarchy = parse_bls_occupation_file(fp, dta_type="industry")
    leaves.to_csv(f"../data/bls/bls_detailed_industry_data_{year}.csv", index=False)
    hierarchy.to_csv(f"../data/bls/bls_industry_data_{year}.csv", index=False)
    
    all_industries = sorted(set(leaves["industry"].to_list()))

    with open(f"../data/industry_{year}.txt", "w") as f:
        f.write("\n".join(all_industries))

In [3]:

fp = f"../data/bls/cpsaat18_2024.xlsx"
leaves, hierarchy = parse_bls_occupation_file(fp, dta_type="industry")

In [12]:
get_full_ancestry(leaves, hierarchy, 'industry')

,industry,total_employed,pct_women,pct_white,pct_black,pct_asian,pct_hispanic,parent_id,major_group,mid_group,minor_group
0,Crop production,1188.0,29.7,90.6,3.4,1.4,35.8,L0_0001,"Agriculture, forestry, fishing, and hunting",None,None
1,Animal production and aquaculture,727.0,30.1,93.2,2.3,0.2,19.4,L0_0001,"Agriculture, forestry, fishing, and hunting",None,None
2,Support activities for agriculture and forestry,152.0,36.4,96.2,0.2,1.3,27.3,L0_0001,"Agriculture, forestry, fishing, and hunting",None,None
3,Forestry except logging,47.0,NaN,NaN,NaN,NaN,NaN,L0_0001,"Agriculture, forestry, fishing, and hunting",None,None
4,Logging,72.0,9.8,92.0,4.9,0.0,6.5,L0_0001,"Agriculture, forestry, fishing, and hunting",None,None
...,...,...,...,...,...,...,...,...,...,...,...
253,"Justice, public order, and safety activities",2665.0,34.3,77.1,15.3,3.1,16.7,L0_0054,Public administration,None,None
254,Administration of human resource programs,1564.0,70.9,64.8,22.0,7.1,16.7,L0_0054,Public administration,None,None
255,"Administration of environmental quality, and h...",265.0,50.2,78.9,7.6,4.8,12.3,L0_0054,Public administration,None,None
256,Administration of economic programs and space ...,604.0,39.3,75.9,13.9,7.0,10.3,L0_0054,Public administration,None,None


### do we need to worry about providing unique job categories for each year?
Compare the occupations files

In [28]:
import hashlib
from pathlib import Path

files = [f"../data/occupations_{year}.txt" for year in range(2017, 2025)]

hashes = {}
for f in files:
    h = hashlib.md5(Path(f).read_bytes()).hexdigest()
    hashes[f] = h

# Group files by hash
groups = {}
for f, h in hashes.items():
    groups.setdefault(h, []).append(f)

if len(groups) == 1:
    print("Hurray, all files are identical")
else:
    for h, members in groups.items():
        print(f"Group {h[:8]}: {members}")

Group a6baeeb2: ['../data/occupations_2017.txt', '../data/occupations_2018.txt', '../data/occupations_2019.txt']
Group 778163c5: ['../data/occupations_2020.txt']
Group f74e53db: ['../data/occupations_2021.txt', '../data/occupations_2022.txt', '../data/occupations_2023.txt', '../data/occupations_2024.txt']


Sadly, things have changed. What's different?

In [30]:
def load_occupations(path):
    return set(Path(path).read_text().splitlines())
from itertools import combinations

for a, b in combinations(files, 2):
    set_a = load_occupations(a)
    set_b = load_occupations(b)
    added = set_b - set_a
    removed = set_a - set_b

    if added or removed:
        print(f"\n{a} vs {b}:")
        for occ in sorted(added):
            print(f"  + {occ}")
        for occ in sorted(removed):
            print(f"  - {occ}")
    else:
        print(f"\n{a} vs {b}: identical")


../data/occupations_2017.txt vs ../data/occupations_2018.txt: identical

../data/occupations_2017.txt vs ../data/occupations_2019.txt: identical

../data/occupations_2017.txt vs ../data/occupations_2020.txt:
  + Acupuncturists
  + Animal caretakers
  + Architects, except landscape and naval
  + Architectural and civil drafters
  + Athletes and sports competitors
  + Audiovisual equipment installers and repairers
  + Bailiffs
  + Bioengineers and biomedical engineers
  + Broadcast announcers and radio disc jockeys
  + Broadcast, sound, and lighting technicians
  + Bus drivers, school
  + Bus drivers, transit and intercity
  + Cardiovascular technologists and technicians
  + Child, family, and school social workers
  + Clinical and counseling psychologists
  + Coaches and scouts
  + Commercial and industrial designers
  + Computer numerically controlled tool operators and programmers
  + Construction equipment operators
  + Conveyor, dredge, and hoist and winch operators
  + Correctiona

__Conclusion__: need to pass in list per year to get the right props